<td>
   <a target="_blank" href="https://labelbox.com" ><img src="https://labelbox.com/blog/content/images/2021/02/logo-v4.svg" width=256/></a>
</td>


<td>
<a href="https://colab.research.google.com/github/Labelbox/labelbox-notebooks/blob/main/project_configuration/multimodal_chat_project.ipynb" target="_blank"><img
src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>
</td>

<td>
<a href="https://github.com/Labelbox/labelbox-notebooks/tree/main/project_configuration/multimodal_chat_project.ipynb" target="_blank"><img
src="https://img.shields.io/badge/GitHub-100000?logo=github&logoColor=white" alt="GitHub"></a>
</td>

# Multimodal chat project setup

This notebook provides an example workflow of setting up a multimodal chat (MMC) project using the Labelbox Python SDK.
MMC projects require the following unique methods and modifications to existing methods:

- `client.create_model_evaluation_project`: The main method to create a live MMC project.
  
- `client.create_offline_model_evaluation_project`: The main method to create an offline MMC project.

- `client.create_ontology`: The method to create ontologies for MNC projects. Requires an `ontology_kind` parameter set to `lb.OntologyKind.ModelEvaluation`.

- `client.create_ontology_from_feature_schemas`: Similar to `client.create_ontology` but takes a list of `feature schema ids` to allow you to use existing features instead of creating new features. Also requires an `ontology_kind` set to `lb.OntologyKind.ModelEvaluation`.

## Set up

In [ ]:
%pip install -q --upgrade "labelbox[data]"

In [ ]:
import labelbox as lb

## API key and client
Provide a valid API key below to connect to the Labelbox client properly. For more information, see [Create API keys](https://docs.labelbox.com/reference/create-api-key).

In [ ]:
API_KEY = ""
client = lb.Client(api_key=API_KEY)

## Example: Create a MMC project

Creating a MMC project using the Labelbox Python SDK follows the same general steps as a regular project, with a few differences. This example workflow highlights those differences.

### Create a MMC ontology

You can create ontologies for multimodal chat projects in the same way as other project ontologies using two methods: `client.create_ontology` and `client.create_ontology_from_feature_schemas`. The only additional requirement is to pass an ontology_kind parameter, which needs to be set to `lb.OntologyKind.ModelEvaluation`.

#### Option A: `client.create_ontology`

Typically, you create ontologies and generate the associated features simultaneously. The following example of creating an ontology for your MMC project using supported tools and classifications. See [Supported annotation types](https://docs.labelbox.com/docs/multimodal-chat-evaluation-editor#supported-annotation-types) for the annotation types you can include in an MMC ontology.

In [ ]:
ontology_builder = lb.OntologyBuilder(
    tools=[
        lb.Tool(
            tool=lb.Tool.Type.MESSAGE_SINGLE_SELECTION,
            name="single select feature",
        ),
        lb.Tool(
            tool=lb.Tool.Type.MESSAGE_MULTI_SELECTION,
            name="multi select feature",
        ),
        lb.Tool(tool=lb.Tool.Type.MESSAGE_RANKING, name="ranking feature"),
    ],
    classifications=[
        lb.Classification(
            class_type=lb.Classification.Type.CHECKLIST,
            name="checklist feature",
            options=[
                lb.Option(value="option 1", label="option 1"),
                lb.Option(value="option 2", label="option 2"),
            ],
        ),
        lb.Classification(
            class_type=lb.Classification.Type.RADIO,
            name="radio_question",
            options=[
                lb.Option(value="first_radio_answer"),
                lb.Option(value="second_radio_answer"),
            ],
        ),
    ],
)

# Create ontology
ontology = client.create_ontology(
    "MMC ontology",
    ontology_builder.asdict(),
    media_type=lb.MediaType.Conversational,
    ontology_kind=lb.OntologyKind.ModelEvaluation,
)

### Option B: `client.create_ontology_from_feature_schemas`
You can create ontologies using feature schema IDs to reuse existing feature schemas instead of generating new ones. To obtain these IDs, go to the **Schema** tab on the web platform.

In [ ]:
ontology = client.create_ontology_from_feature_schemas(
    "MMC ontology",
    feature_schema_ids=["<list of feature schema ids"],
    media_type=lb.MediaType.Conversational,
    ontology_kind=lb.OntologyKind.ModelEvaluation,
)

## Create MMC projects

The two types of MMC projects have different project creation methods and data row setups:

- For **offline multimodal chat evaluation** projects, use `create_offline_model_evaluation_project` and import data rows of existing conversations.

- For **live multimodal chat evaluation projects**, use `client.create_model_evaluation_project` and either:
  - (Recommended) Create data rows and send them to projects, like other types of projects.
  - Generate empty data rows upon project creation, which can't create data rows with attachments and metadata.

## Set up offline MMC projects

Use `client.create_offline_model_evaluation_project` to create offline multimodal chat evaluation projects. This method uses the same parameters as `client.create_project` and adds validation to ensure the project is set up correctly.

In [ ]:
project = client.create_offline_model_evaluation_project(
    name="<project_name>",
    description="<project_description>",  # optional
)

### Set up live MMC projects

Use `client.create_model_evaluation_project` to create a live multimodal chat evaluation project. This method takes the same parameters as the traditional `client.create_project`, with a few additional parameters specific to multimodal chat evaluation projects.

The `client.create_model_evaluation_project` methods require the following parameters:

- `name`: The name of your new project.

- `description` (optional): The description of your project.

- `data_row_count` (optional): The number of data rows to generate for your project. Defaults to 100 if a `dataset_name` or `dataset_id` is included.

- `dataset_name` (optional): The name of a new dataset. Include this parameter only if you want to create a new dataset for the generated data rows.

- `dataset_id` (optional): The dataset ID of an existing Labelbox dataset. Include this parameter only if you want to append generated data rows to an existing dataset.


In [ ]:
# Create the project
project = client.create_model_evaluation_project(
    name="Example live multimodal chat project",
    description="<project_description>",  # optional
)

def make_data_rows(dataset_id=None):
    # If a dataset ID is provided, fetch the dataset using that ID.
    # Otherwise, create a new dataset with the specified name.
    if dataset_id:
        dataset = client.get_dataset(dataset_id)
    else:
        dataset = client.create_dataset(name="example live mmc dataset")

    # Helper function to generate a single data row
    def generate_data(ind):
        return {
            "row_data": {  # The chat data format
                'type': 'application/vnd.labelbox.conversational.model-chat-evaluation',
                'draft': True,
                'rootMessageIds': [],
                'actors': {},
                'version': 2,
                'messages': {}
            },
            "global_key": f"global_key_{dataset.uid}_{ind}",
            "metadata_fields": [{"name": "tag", "value": "val_tag"}],
            "attachments": [
                {
                    "type": "IMAGE_OVERLAY",
                    "value": "https://storage.googleapis.com/labelbox-sample-datasets/Docs/rgb.jpg"
                }
            ]
        }

    # Generate a list of 100 data rows
    data_list = [generate_data(ind) for ind in range(100)]

    # Upload the generated data rows to the dataset
    task = dataset.create_data_rows(data_list)
    print("Processing task ", task.uid)  # Print the unique ID of the task
    task.wait_till_done()
    
    # Ensure that the task status is 'COMPLETE' to confirm success
    assert task.status == "COMPLETE"

    # Return the dataset object
    return dataset

# Create a new data set. Alternatively, pass an existing dataset ID
dataset = make_data_rows()

# Retrieve the data row IDs from the dataset
data_row_ids = [data_row.uid for data_row in dataset.data_rows()]

# Send data rows to the project
batch = project.create_batch(
    name="mmc-batch",  # each batch in a project must have a unique name
    data_rows=data_row_ids, # data row IDs to include in the batch
    priority=1  # priority between 1(highest) - 5(lowest)
)

print(f"Batch: {batch}")

## Set up model config
You can create, delete, attach, and remove model configs for evaluating your live MMC responses using the SDK.


### Create model configs

The method for creating a model config is `client.create_model_config`. It takes the following parameters:

- `name`: The name of the model config.

- `model_id`: The ID of the model to configure. To get this value, go to the Model tab, select your model, and copy the ID from the URL.

- `inference_params`: Model configuration parameters in the JSON format. Each model has unique parameters.

In [ ]:
MODEL_ID = "270a24ba-b983-40d6-9a1f-98a1bbc2fb65"

inference_params = {"max_new_tokens": 1024, "use_attachments": True}

model_config = client.create_model_config(
    name="Example model config",
    model_id=MODEL_ID,
    inference_params=inference_params,
)

### Attach model config to project
You can attach and remove model configs to your project using `project.add_model_config` or `project.remove_model_config`. Both methods take just a `model_config` ID.

In [ ]:
project.add_model_config(model_config.uid)

### Delete model config
Use `project.delete_project_model_config()` or `client.delete_model_config` to delete model configs. Both methods require the `model_config` ID as a parameter. You can obtain this ID in one of the following ways:

- From the response when you create a model config
- By accessing `project.project_model_configs` and iterating through the list of model configs attached to your project

In [ ]:
model_configs = project.project_model_configs()

for model_config in model_configs:
    client.delete_model_config(model_config.uid)

### Mark project setup as complete

Once you have finalized your project and set up your model configs, you must mark the project setup as complete.

**Once the project is marked as "setup complete", a user can not add, modify, or delete existing project model configs.**

In [ ]:
project.set_project_model_setup_complete()

## Clean up

This section serves as an optional clean-up step to delete the Labelbox assets created within this guide.

In [ ]:
project.delete()
client.delete_unused_ontology(ontology.uid)
dataset.delete()